# Donor Retention Risk - Notebook

Interactive notebook version of the donor churn pipeline.
It trains a model, evaluates it, and exports donor risk scores.

In [1]:
from pathlib import Path
import sys
import json
import math
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import load_raw_data, build_features, apply_log_transforms
from src.modeling import build_preprocessor, fit_best_model, permutation_feature_importance
from src.eda import univariate_summary, bivariate_analysis

DATA_DIR = PROJECT_ROOT / "data"
OUT_DIR = PROJECT_ROOT / "outputs" / "donor_retention"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHURN_WINDOW_DAYS = 180
PRIORITIZE = "recall"   # or "roc_auc"
TEST_SIZE = 0.2
TUNE = True
SKEW_THRESHOLD = 1.0

print("Setup complete")
print(f"DATA_DIR: {DATA_DIR}")
print(f"OUT_DIR : {OUT_DIR}")

Setup complete
DATA_DIR: /Users/matthewvance/Desktop/Intex2Prep/Intex2/ml-pipelines/data
OUT_DIR : /Users/matthewvance/Desktop/Intex2Prep/Intex2/ml-pipelines/outputs/donor_retention


In [2]:
donations, supporters, donation_allocations, social_posts = load_raw_data(str(DATA_DIR))
X, y, meta = build_features(
    donations=donations,
    supporters=supporters,
    donation_allocations=donation_allocations,
    social_posts=social_posts,
    churn_window_days=CHURN_WINDOW_DAYS,
)

print(f"Feature rows: {len(X)}")
print(f"Positive rate: {y.mean():.3f}")
X.head()

Feature rows: 59
Positive rate: 0.339


,total_donations,avg_donation_amount,total_donation_amount,days_since_last_donation,days_since_first_donation,donation_frequency,pct_recurring,num_posts,engagement_score,supporter_type,relationship_type,region,country,acquisition_channel,status
supporter_id,,,,,,,,,,,,,,,
1,12,1081.138571,7567.97,10.0,1072.0,4.088619,1.0,3,9.0,SocialMediaAdvocate,Local,Luzon,Philippines,SocialMedia,Active
2,4,1740.040000,3480.08,297.0,1089.0,1.341598,0.0,1,3.0,Volunteer,Local,Mindanao,Philippines,SocialMedia,Active
3,16,1025.078889,9225.71,169.0,1103.0,5.298277,1.0,1,3.0,MonetaryDonor,Local,Luzon,Philippines,SocialMedia,Active
4,11,1086.841250,8694.73,0.0,1082.0,3.713262,1.0,3,9.0,MonetaryDonor,PartnerOrganization,Mindanao,Philippines,Church,Active
5,5,1184.645000,4738.58,150.0,802.0,2.277120,0.0,1,3.0,InKindDonor,PartnerOrganization,Mindanao,Philippines,Website,Active


In [3]:
eda_df = univariate_summary(X)
eda_path = OUT_DIR / "eda_univariate_summary.csv"
eda_df.to_csv(eda_path, index=False)
print(f"Saved -> {eda_path}")
eda_df.head(20)

Saved -> /Users/matthewvance/Desktop/Intex2Prep/Intex2/ml-pipelines/outputs/donor_retention/eda_univariate_summary.csv


,column,dtype,n_missing,pct_missing,n_unique,mean,std,min,p25,median,p75,max,skew,top_value
0,total_donations,int64,0,0.00,17,7.1186,4.5147,1.0000,4.000,6.0000,9.0000,23.0000,1.5582,NaN
1,avg_donation_amount,float64,2,3.39,57,1069.4565,504.8383,250.0000,705.180,990.8425,1255.0450,3014.8675,1.6203,NaN
2,total_donation_amount,float64,0,0.00,58,4080.0769,3093.9109,0.0000,1869.805,3157.7600,5916.6550,12059.4700,0.8552,NaN
3,days_since_last_donation,float64,0,0.00,56,185.5424,186.8525,0.0000,59.500,118.0000,267.5000,797.0000,1.5928,NaN
4,days_since_first_donation,float64,0,0.00,56,951.8475,195.7826,388.0000,827.000,1025.0000,1091.0000,1147.0000,-1.3181,NaN
5,donation_frequency,float64,0,0.00,57,2.6863,1.4718,0.4583,1.778,2.2930,3.2239,7.7498,1.3911,NaN
6,pct_recurring,float64,0,0.00,2,0.3051,0.4644,0.0000,0.000,0.0000,1.0000,1.0000,0.8689,NaN
7,num_posts,int64,0,0.00,6,1.3051,1.2213,0.0000,0.000,1.0000,2.0000,5.0000,0.7950,NaN
8,engagement_score,float64,0,0.00,6,3.9153,3.6638,0.0000,0.000,3.0000,6.0000,15.0000,0.7950,NaN
9,supporter_type,str,0,0.00,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MonetaryDonor


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=42
)
print(f"Train: {len(X_train)} | Test: {len(X_test)} | Train churn rate: {y_train.mean():.3f}")

biv_df = bivariate_analysis(X_train, y_train)
biv_path = OUT_DIR / "eda_bivariate_analysis.csv"
biv_df.to_csv(biv_path, index=False)
print(f"Saved -> {biv_path}")
biv_df.head(20)

Train: 47 | Test: 12 | Train churn rate: 0.340
Saved -> /Users/matthewvance/Desktop/Intex2Prep/Intex2/ml-pipelines/outputs/donor_retention/eda_bivariate_analysis.csv


,feature,dtype,stat_type,stat_value,p_value
0,total_donations,int64,pearson_r,-0.3453,0.0174
1,avg_donation_amount,float64,pearson_r,0.1324,0.3860
2,total_donation_amount,float64,pearson_r,-0.2076,0.1614
3,days_since_last_donation,float64,pearson_r,0.8261,0.0000
4,days_since_first_donation,float64,pearson_r,-0.0624,0.6767
5,donation_frequency,float64,pearson_r,-0.4055,0.0047
6,pct_recurring,float64,pearson_r,-0.2435,0.0991
7,num_posts,int64,pearson_r,-0.4023,0.0051
8,engagement_score,float64,pearson_r,-0.4023,0.0051
9,supporter_type,str,chi2,3.7019,0.5931


In [5]:
X_train, transformed_cols = apply_log_transforms(X_train, skew_threshold=SKEW_THRESHOLD)
if transformed_cols:
    X_test = X_test.copy()
    X_test[transformed_cols] = np.log1p(X_test[transformed_cols])
print(f"Log-transformed cols: {transformed_cols if transformed_cols else 'None'}")

prep, _, _ = build_preprocessor(X_train)

neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
pos_weight = neg / pos if pos > 0 else 1.0
print(f"Class counts -> negative: {neg}, positive: {pos}, pos_weight: {pos_weight:.2f}")

Log-transformed cols: ['total_donations', 'avg_donation_amount', 'days_since_last_donation', 'days_since_first_donation', 'donation_frequency', 'pct_recurring']
Class counts -> negative: 31, positive: 16, pos_weight: 1.94


In [6]:
best_name, pipe, test_metrics, eval_df, tune_info = fit_best_model(
    X_train=X_train,
    y_train=y_train,
    preprocessor=prep,
    X_test=X_test,
    y_test=y_test,
    prioritize=PRIORITIZE,
    tune=TUNE,
    pos_weight=pos_weight,
)

print(f"Best model: {best_name}")
print(f"Held-out test metrics: {test_metrics}")
if tune_info:
    print(f"Tuning info: {tune_info}")
eval_df

Best model: random_forest
Held-out test metrics: {'roc_auc': 1.0, 'pr_auc': 1.0, 'recall_at_50': 1.0}
Tuning info: {'best_params': {'clf__n_estimators': 300, 'clf__min_samples_leaf': 4, 'clf__max_depth': None}, 'best_cv_score': 0.8166666666666667}


,model,roc_auc_mean,roc_auc_std,recall_mean,recall_std
0,random_forest,0.966667,0.044444,0.750000,0.247207
1,gradient_boosting,1.000000,0.000000,1.000000,0.000000
2,decision_tree,1.000000,0.000000,1.000000,0.000000
3,log_reg,0.847619,0.097254,0.666667,0.210819


In [7]:
imp_df = permutation_feature_importance(pipe, X_test, y_test, n_repeats=10)
imp_path = OUT_DIR / "feature_importance.csv"
imp_df.to_csv(imp_path, index=False)
print(f"Saved -> {imp_path}")
imp_df.head(20)

Saved -> /Users/matthewvance/Desktop/Intex2Prep/Intex2/ml-pipelines/outputs/donor_retention/feature_importance.csv


,feature,importance_mean,importance_std
3,days_since_last_donation,0.265625,0.071603
0,total_donations,0.000000,0.000000
1,avg_donation_amount,0.000000,0.000000
2,total_donation_amount,0.000000,0.000000
4,days_since_first_donation,0.000000,0.000000
5,donation_frequency,0.000000,0.000000
6,pct_recurring,0.000000,0.000000
7,num_posts,0.000000,0.000000
8,engagement_score,0.000000,0.000000
9,supporter_type,0.000000,0.000000


In [8]:
model_path = OUT_DIR / f"donor_churn_{best_name}.joblib"
joblib.dump(pipe, model_path)
print(f"Saved model -> {model_path}")

X_full = X.copy()
if transformed_cols:
    X_full[transformed_cols] = np.log1p(X_full[transformed_cols])

probas = pipe.predict_proba(X_full)[:, 1]
result = pd.DataFrame({
    "supporter_id": X_full.index,
    "churn_risk": probas,
    "label_churn": y.values,
})
threshold = result["churn_risk"].quantile(0.80)
result["at_risk_top20"] = (result["churn_risk"] >= threshold).astype(int)

def personalized_action(x_row: pd.Series, meta_row) -> tuple:
    avg_amt = float(x_row.get("avg_donation_amount", 0) or 0)
    freq = float(x_row.get("donation_frequency", 0) or 0)
    recency_days = float(x_row.get("days_since_last_donation", 9999) or 9999)
    eng = float(x_row.get("engagement_score", 0) or 0)
    num_posts = float(x_row.get("num_posts", 0) or 0)
    cause = None
    if meta_row is not None and "cause_focus" in meta_row and pd.notna(meta_row["cause_focus"]) and meta_row["cause_focus"] != "unknown":
        cause = str(meta_row["cause_focus"])

    high_value = avg_amt >= 250
    mid_value = 75 <= avg_amt < 250
    high_freq = freq >= 12
    mid_freq = 4 <= freq < 12
    recently_lapsed = 180 <= recency_days < 270
    long_lapsed = recency_days >= 270
    engaged = (eng >= 10) or (num_posts >= 3)

    if high_value and long_lapsed:
        return (f"Director call + tailored {cause or 'impact'} report; invite to briefing", f"High average gift (${avg_amt:.0f}) and long lapse ({int(recency_days)} days).")
    if high_value and recently_lapsed:
        return (f"Handwritten note + {cause or 'impact'} update; offer matched-gift opportunity", f"High average gift (${avg_amt:.0f}) and recent lapse ({int(recency_days)} days).")
    if engaged and recently_lapsed:
        return (f"DM on social + story from {cause or 'their top cause'}; link to quick donate", f"High engagement (score {eng:.0f}) with recent lapse ({int(recency_days)} days).")
    if engaged and long_lapsed:
        return (f"Re-engagement series highlighting recent {cause or 'program'} wins; invite to event", f"High engagement (score {eng:.0f}) but long lapse ({int(recency_days)} days).")
    if high_freq and recently_lapsed:
        return (f"Set-it-and-forget-it monthly reminder; show {cause or 'impact'} progress bar", f"Historically frequent donor (~{freq:.1f}/yr) with recent lapse ({int(recency_days)} days).")
    if mid_freq and mid_value:
        return (f"Upsell to recurring gift with {cause or 'impact'} milestone; thank-you email", f"Moderate frequency (~{freq:.1f}/yr) and mid average gift (${avg_amt:.0f}).")
    if mid_value and long_lapsed:
        return (f"Limited-time match for {cause or 'priority program'}; show before/after outcomes", f"Mid average gift (${avg_amt:.0f}) with long lapse ({int(recency_days)} days).")
    if avg_amt < 50 and long_lapsed:
        return ("Low-friction 1-click $15 ask + gratitude; option to choose favorite cause", f"Low average gift (${avg_amt:.0f}) and long lapse ({int(recency_days)} days).")
    if avg_amt < 50 and recently_lapsed:
        return ("Friendly reminder with concrete $ impact examples; suggest small recurring gift", f"Low average gift (${avg_amt:.0f}) with recent lapse ({int(recency_days)} days).")
    if cause:
        return (f"Send {cause} success story + evidence of impact; thank and invite feedback", f"Donor shows preference for {cause}.")
    return ("Personalized email with recent impact story + easy donate link", "General follow-up due to risk and limited preference signals.")

actions, whys = [], []
for sid in result["supporter_id"]:
    row_meta = meta.loc[sid] if sid in meta.index else None
    x_row = X_full.loc[sid] if sid in X_full.index else pd.Series()
    action, why = personalized_action(x_row, row_meta)
    actions.append(action)
    whys.append(why)

result["suggested_action"] = actions
result["suggested_why"] = whys

scores_path = OUT_DIR / "donor_risk_scores.parquet"
result.to_parquet(scores_path, index=False)
print(f"Saved scores -> {scores_path}")

def _safe_json(obj):
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return None
        return obj
    if isinstance(obj, dict):
        return {k: _safe_json(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_safe_json(v) for v in obj]
    return obj

metrics_payload = {
    "model_name": best_name,
    "test_metrics": test_metrics,
    "tune_info": tune_info,
    "cv_metrics": eval_df.to_dict(orient="records"),
    "log_transformed_cols": transformed_cols,
    "churn_window_days": CHURN_WINDOW_DAYS,
    "prioritize": PRIORITIZE,
    "train_size": len(X_train),
    "test_size_n": len(X_test),
}
metrics_path = OUT_DIR / "metrics.json"
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(_safe_json(metrics_payload), f, indent=2)
print(f"Saved metrics -> {metrics_path}")

result.sort_values("churn_risk", ascending=False).head(20)

Saved model -> /Users/matthewvance/Desktop/Intex2Prep/Intex2/ml-pipelines/outputs/donor_retention/donor_churn_random_forest.joblib


Saved scores -> /Users/matthewvance/Desktop/Intex2Prep/Intex2/ml-pipelines/outputs/donor_retention/donor_risk_scores.parquet
Saved metrics -> /Users/matthewvance/Desktop/Intex2Prep/Intex2/ml-pipelines/outputs/donor_retention/metrics.json


,supporter_id,churn_risk,label_churn,at_risk_top20,suggested_action,suggested_why
51,53,0.871208,1,1,Send Transport success story + evidence of imp...,Donor shows preference for Transport.
38,40,0.853775,1,1,Send Outreach success story + evidence of impa...,Donor shows preference for Outreach.
13,14,0.853588,1,1,Send Education success story + evidence of imp...,Donor shows preference for Education.
14,15,0.850587,1,1,Send Operations success story + evidence of im...,Donor shows preference for Operations.
56,58,0.838657,1,1,Send Outreach success story + evidence of impa...,Donor shows preference for Outreach.
39,41,0.833545,1,1,Send Operations success story + evidence of im...,Donor shows preference for Operations.
22,23,0.824071,1,1,Send Wellbeing success story + evidence of imp...,Donor shows preference for Wellbeing.
1,2,0.807509,1,1,Send Education success story + evidence of imp...,Donor shows preference for Education.
35,37,0.767763,1,1,Send Outreach success story + evidence of impa...,Donor shows preference for Outreach.
34,36,0.755054,1,1,Send Maintenance success story + evidence of i...,Donor shows preference for Maintenance.
